# Chuẩn bị Qwen3.5 + thư viện cho Kaggle (chạy MỘT lần)
Notebook này lưu sẵn **Qwen3.5-9B phần text, đã lượng tử hoá NF4** (đúng cấu hình inference dùng) và các **wheel Python** mà image Kaggle còn thiếu, vào `/kaggle/working`.
Notebook inference gắn output này làm input nên **không cần tải model, không cần lượng tử hoá lại, không cần cài gói qua mạng**.

1. Session options: **Accelerator GPU T4** (x1 hoặc x2), **Internet On**. Không cần gắn dataset.
2. **Save Version → Save & Run All (Commit)** để output được giữ lại (~7 GB; khoảng 10–20 phút: tải ~19 GB, lượng tử hoá, load lại để kiểm tra).
3. Ở notebook inference: **Add Input → Your Work → Notebooks →** chọn notebook này. Notebook inference tự tìm model và wheel.

Chỉ chạy lại khi đổi model/cách lượng tử hoá, hoặc khi Kaggle đổi image (notebook inference sẽ báo nếu wheel không còn khớp).

In [ ]:
REPO_URL = "https://github.com/KhoaNotFound/ZAIC2026-BackpropEverytTime.git"
MODEL_ID = "Qwen/Qwen3.5-9B"
QUANTIZATION = "4bit"  # "none" = FP16 phần text (~18 GB, sát giới hạn 20 GB của /kaggle/working)
MODEL_OUT = "/kaggle/working/qwen3.5-9b-" + ("nf4" if QUANTIZATION == "4bit" else "fp16")
WHEELS_OUT = "/kaggle/working/wheels"
REPO = "/tmp/zAIC"  # ngoài /kaggle/working để output chỉ gồm model + wheel

In [ ]:
import subprocess, sys
from pathlib import Path
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# Tải đúng các gói image này còn thiếu, rồi cài chúng OFFLINE như notebook inference sẽ làm.
subprocess.run([sys.executable, f"{REPO}/scripts/download_wheels.py",
                "--requirements", f"{REPO}/requirements.txt", "--output", WHEELS_OUT], check=True)
wheels = [str(p) for p in Path(WHEELS_OUT).iterdir() if p.suffix in (".whl", ".gz", ".zip")]
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps", *wheels], check=True)

In [ ]:
import torch
print("PyTorch:", torch.__version__, "GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
# Lưu model rồi load lại đúng như inference và sinh thử một câu trả lời.
subprocess.run([sys.executable, f"{REPO}/scripts/prepare_model.py", "--model", MODEL_ID,
                "--quantization", QUANTIZATION, "--output", MODEL_OUT, "--verify"], check=True, cwd=REPO)

In [ ]:
for d in [MODEL_OUT, WHEELS_OUT]:
    size = sum(p.stat().st_size for p in Path(d).rglob("*") if p.is_file())
    print(f"{d}: {size / 2**30:.2f} GiB")
print(Path(MODEL_OUT, "zaic_model.json").read_text())